# 01 · Exploratory Data Analysis: the standard process

### Session 1 support notebook | Safiri Telecom churn case

---

Every analyst invents their own way of looking at a new dataset, forgets half of it
under pressure, and misses something obvious. The fix is a **ritual**: the same
sequence, every time, in the same order.

This notebook gives you that ritual, and one discipline that most courses omit.

> **You split before you explore.**

Looking at the test set is not a technical error that a library will catch. It is a
discipline violation, and it quietly inflates every number you report afterwards.
The test set represents the future. You do not get to look at the future.

### Cell tags

| Tag | What you do |
|---|---|
| **READ** | Run it, read the output, answer the question. No typing. |
| **FILL** | Change one clearly marked value. |
| **BUILD** | Open-ended. Write your own code. |

Everyone completes READ and FILL. If a cell errors, skip it and keep moving.

## 0. Setup and data

In [ ]:
import numpy as np
import pandas as pd

def make_panel(seed=42, n_subscribers=9000, n_months=8):
    """Simulate a monthly prepaid subscriber panel for Safiri Telecom."""
    rng = np.random.default_rng(seed)
    n = n_subscribers
    region  = rng.choice(["Nairobi","Coast","Western","Rift Valley","Eastern"], n, p=[.34,.16,.18,.20,.12])
    plan    = rng.choice(["Prepaid Basic","Prepaid Plus","Data Bundle"], n, p=[.55,.28,.17])
    handset = rng.choice(["Feature","Smart-Entry","Smart-Mid","Smart-High"], n, p=[.22,.38,.28,.12])
    tenure0, base_arpu = rng.integers(1,72,n), rng.lognormal(5.6,.55,n)
    heavy_data, eps = rng.random(n) < .30, rng.normal(0,.55,n)

    r1 = (-1.0 - 0.032*np.clip(tenure0,0,48) - 0.0014*base_arpu
          + 0.60*(plan=="Prepaid Basic") + 0.50*(handset=="Feature")
          + 0.30*(region=="Western") + eps)
    r2 = (-0.55 - 0.006*np.clip(tenure0,0,48) + 0.0003*base_arpu + 1.15*heavy_data
          + 0.70*np.isin(handset,["Smart-Mid","Smart-High"]) + 0.20*(plan=="Prepaid Basic") + eps)
    p1, p2 = 1/(1+np.exp(-r1)), 1/(1+np.exp(-r2))

    churn_month = np.full(n, 99)
    for m in range(1, n_months+1):
        hz = np.clip((p1 if m <= 5 else p2)*0.20, 0, .9)
        churn_month[(rng.random(n) < hz) & (churn_month == 99)] = m

    rows = []
    for m in range(1, n_months+1):
        alive = churn_month >= m
        decay = np.where(np.clip(churn_month-m,0,6) <= 2, 0.66 + 0.17*np.clip(churn_month-m,0,6), 1.0)
        rows.append(pd.DataFrame({
            "subscriber_id": np.arange(n), "month": m, "region": region, "plan": plan,
            "handset_class": handset, "tenure_months": tenure0 + m,
            "topup_kes": np.where(alive, base_arpu*decay*rng.lognormal(0,.30,n), 0).round(0),
            "topup_count": np.where(alive, rng.poisson(np.clip(6*decay,0.3,None)), 0),
            "data_mb": np.where(alive, base_arpu*decay*rng.lognormal(0,.5,n)*(1+1.4*heavy_data), 0).round(1),
            "voice_minutes": np.where(alive, rng.gamma(3, 22*decay), 0).round(1),
            "days_since_last_topup": np.where(alive,
                np.clip(rng.poisson(np.clip(20*(1-decay)+4,1,None)),0,60), 60),
            "complaints_90d": rng.poisson(0.18 + 0.45*((churn_month-m) <= 2), n),
            "active": alive.astype(int), "_churn_month": churn_month}))
    return pd.concat(rows, ignore_index=True)

def build_snapshot(panel, prediction_month, horizon=2):
    """Features as at the prediction point; label from the outcome window."""
    obs = panel[(panel.month == prediction_month) & (panel.active == 1)].copy()
    cm = obs._churn_month.values
    obs["churned"] = ((cm > prediction_month) & (cm <= prediction_month + horizon)).astype(int)
    return obs.drop(columns=["_churn_month", "active", "month"])

FEATURES_NUM = ["tenure_months","topup_kes","topup_count","data_mb",
                "voice_minutes","days_since_last_topup","complaints_90d"]
FEATURES_CAT = ["region","plan","handset_class"]

import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 150)
plt.rcParams["figure.figsize"] = (10, 4.5)

panel = make_panel()
print(f"Panel: {panel.shape[0]:,} subscriber-months")

## 1. Split first

Before a single `describe()`, before a single plot. Train on the past, test on the
future, exactly as production will work.

In [ ]:
train = pd.concat([build_snapshot(panel, 3), build_snapshot(panel, 4)], ignore_index=True)
test  = build_snapshot(panel, 6)

# From here on, `test` is sealed. We do not describe it, plot it, or look at it.
print(f"TRAIN (months 3-4): {len(train):,} rows")
print(f"TEST  (month 6)   : {len(test):,} rows   <-- SEALED")

df = train.copy()      # everything below uses the training data only

---

# The ritual

Nine steps, always in this order. Steps 1 to 4 tell you what you have. Steps 5 to 8
tell you what is wrong with it. Step 9 tells you what to do next.

## Step 1 · Shape. Did the right data arrive?

In [ ]:
print("Rows, columns:", df.shape)
print("Duplicate subscriber rows:", df.subscriber_id.duplicated().sum())

**READ.** That duplicate count is not zero, and it should not alarm you: the same
subscriber appears in both the month 3 and month 4 snapshots.

But it is exactly the condition that causes **group leakage**. If you ever split
these rows at random, the same person lands in both train and test and the model
memorises them. Our temporal split avoids it here. Remember the check.

## Step 2 · Types and completeness

In [ ]:
df.info()

## Step 3 · Eyeball the actual rows

In [ ]:
df.head()

Never skip this. Summary statistics hide things that one look at raw rows reveals:
a column that is entirely zero, a date stored as text, an ID that is not unique, a
currency figure that is obviously in the wrong unit.

## Step 4 · Summary statistics, numeric and categorical

In [ ]:
df.describe().T

In [ ]:
df.describe(include="object").T

## Step 5 · Missing values

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0]
print("Columns with missing values:" if len(missing) else "No missing values.")
print(missing.to_string() if len(missing) else "")

## Step 6 · Numeric distributions

In [ ]:
df[FEATURES_NUM].hist(figsize=(13, 7), bins=30, color="#0F766E", edgecolor="white")
plt.tight_layout(); plt.show()

**READ and answer.** Look at `days_since_last_topup`.

> *What is that spike at the far right, and what does it tell you about how the
> data was assembled?*

Answers to shape questions like this almost always live with the person who built
the table, not in the table.

## Step 7 · Categorical distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, col in zip(axes, FEATURES_CAT):
    order = df[col].value_counts().index
    sns.countplot(y=col, data=df, order=order, ax=ax, color="#0F766E")
    ax.set_title(col); ax.set_ylabel("")
plt.tight_layout(); plt.show()

This is where **structural problems** surface: the same category spelled three ways,
a level with four records, a placeholder like `Unknown` that should be missing.
Notebook 02 fixes them. Here you only find them.

## Step 8 · Outliers, then segmentation against the target

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
for ax, col in zip(axes, ["topup_kes", "data_mb", "voice_minutes"]):
    sns.violinplot(x=df[col], ax=ax, color="#DFF7F5")
    ax.set_title(col)
plt.tight_layout(); plt.show()

In [ ]:
# Segmentation is the step that actually generates hypotheses.
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4))
for ax, col in zip(axes, ["tenure_months", "topup_kes", "data_mb", "complaints_90d"]):
    sns.violinplot(x=df[col], y=df.churned.map({0: "stayed", 1: "churned"}),
                   ax=ax, palette=["#94A3B8", "#0F766E"], hue=df.churned.map({0:"stayed",1:"churned"}),
                   legend=False)
    ax.set_title(col); ax.set_ylabel("")
plt.tight_layout(); plt.show()

## Step 9 · Correlations, with a warning

In [ ]:
# numeric_only=True is REQUIRED on pandas 2.0+. Without it this raises ValueError
# on any frame containing text columns.
corr = df[FEATURES_NUM + ["churned"]].corr(numeric_only=True)

plt.figure(figsize=(7.5, 6))
sns.heatmap(corr, cmap="RdBu_r", center=0, annot=True, fmt=".2f",
            annot_kws={"size": 9}, cbar_kws={"shrink": .8})
plt.title("Correlation, training data only"); plt.tight_layout(); plt.show()

**Two warnings about this plot, and both matter more than the plot.**

**Correlation measures straight lines only.** A feature with a correlation of 0.02
can be highly predictive if the relationship is a curve or a threshold. Never drop
a feature on the strength of this heatmap.

**A high correlation with the target is a reason for suspicion, not celebration.**
In the lifecycle notebook, the leaked columns were the most correlated ones in the
table. When a feature looks unusually strong, your first move is to check where it
comes from and when it is populated, not to build with it.

---

# From exploration to hypotheses

EDA that does not end in a written hypothesis was entertainment. Three rules for
a usable hypothesis: it names a mechanism, it is falsifiable, and it implies an
action somebody could take.

In [ ]:
# BUILD: write three hypotheses from what you saw above.
# Format: "<segment> churn more because <mechanism>, so we should <action>."

HYPOTHESES = [
    "...",
    "...",
    "...",
]
for i, h in enumerate(HYPOTHESES, 1):
    print(f"{i}. {h}")

### Checkpoint

You have not once touched `test`. That restraint is the whole lesson of this
notebook, and it costs you nothing: every insight above came from the training data,
which is all you would have had in real life anyway.

**Next:** `02_Cleaning_and_Enrichment` fixes what you found here.